# Intro

## Global parameters

In [1]:
MAX_TIME_HOURS=1
ALPHA=0.01
_PVAL_FLOOR=10**-6

## Modules

### Standard

In [2]:
import os, pickle, platform, sys
import numpy as np
import torch

In [3]:
from collections import defaultdict

In [4]:
import dcms
from dcms.models import DCMModel, DECMModel, qDECMModel, DWCMModel

In [5]:
import matplotlib.pyplot as plt
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['xtick.major.width'] = 2
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['ytick.major.width'] = 2

plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

plt.rcParams['xtick.minor.size'] = 5
plt.rcParams['xtick.minor.width'] = 1
plt.rcParams['ytick.minor.size'] = 5
plt.rcParams['ytick.minor.width'] = 1
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

In [6]:
from scipy.stats import spearmanr

In [7]:
from tqdm.notebook import tqdm, trange

In [8]:
import datetime as dt

In [9]:
from bowtie import edges2bowtie

### Home made

In [10]:
if platform.system() == 'Darwin':
    print('Air!')
    HOME = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2/'
elif platform.system() == 'Linux':
    print('Stella!')
    HOME = '/home/sarawalk/bowtie2_py39/bowtie2/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")

sys.path.insert(0, HOME)

Air!


In [11]:
from auxiliary_functions import el2ks, bic

In [12]:
from sam_bowtie import block_and_fluxes as bnf

In [13]:
from bowtie_plot_functions import plot_bowtie_blocks, plot_bowtie_fluxes, _add_colorbar
from bowtie_plot_functions import _fdr as fdr

## Load data

In [14]:
DATA_FOLDER=HOME+'dati_elezioni/'
TEST_FOLDER=HOME+'tests/'
PVALUE_FOLDER=HOME+'pvalues/'
GUARINO_FOLDER=HOME+'guarino_files/'
BIPARTITE_FOLDER=HOME+'BiDCM/'
PLOT_FOLDER=HOME+'plots/'

# Where are we?

In [59]:
os.listdir(TEST_FOLDER)

['quirinale_dico0_dwcm_final.pkl',
 'crisi_dico3_qdecm.pkl',
 'batch2608_ita_elections_dico6_decm.pkl',
 'decm_validation',
 'ita_elections_dico3_dwcm_final.pkl',
 'crisi_run9.log',
 'crisi_dico0_dwcm_final.pkl',
 'crisi_dico4_decm.pkl',
 'ita_elections_dico4_dwcm_final.pkl',
 'crisis_qdecm_new_theta_nprocs_8.pkl',
 'batch2608_quirinale_dico4_decm.pkl',
 'ita_elections_dico5_dwcm_final.pkl',
 'crisi_dico1_dwcm_final.pkl',
 'quirinale_dico6_dwcm_final.pkl',
 'ita_elections_dico2_dwcm_final.pkl',
 'quirinale_dico1_qdecm.pkl',
 'quirinale_dico1_dwcm_final.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_0_gauge_min.pkl',
 'ita_elections_dico1_qdecm.pkl',
 'ita_elections_dico0_dwcm_final.pkl',
 'batch2608_quirinale_dico2_decm.pkl',
 'batch2608_quirinale_dico3_decm.pkl',
 'ita_elections_dico3_decm_stella.pkl',
 'crisi_dico4_dwcm_final.pkl',
 'quirinale_dico3_dwcm_final.pkl',
 'crisi_dico2_qdecm.pkl',
 'crisi_run_symgs.log',
 'ita_elections_dico6_qdecm.pkl',
 'quirinale_dico6_qdecm.pkl',
 'cris

In [60]:
solution_files=[f for f in os.listdir(TEST_FOLDER) if f.endswith('.pkl')]
solution_files.sort()

In [61]:
final_dict={}
for solution in solution_files:
    # get the model type from the solution filename
    if 'qdecm' in solution.lower():
        model_type='qDECM'
    elif 'decm' in solution.lower():
        model_type='DECM'
    elif 'dwcm' in solution.lower():
        model_type='DWCM'
    else:
        print(f"Unknown model type in solution {solution}. Skipping.")
        continue

    # get the dataset from the solution filename
    if 'elections' in solution.lower():
        name='i'
    elif 'crisi' in solution.lower():
        name='c'
    elif 'quirinale' in solution.lower():
        name='q'
    else:
        print(f"Unknown dataset in solution {solution}. Skipping.")
        continue
    
    try:
        i_dico=solution.index('dico')
        dico=solution[i_dico+4]
    except ValueError:
        print(f"No 'dico' found in solution {solution}. Skipping.")
        continue

    dataset=f"{name}{dico}"
    if os.path.exists(TEST_FOLDER+solution):
        try:
            with open(TEST_FOLDER+solution, 'rb') as f:
                _temp = pickle.load(f)
        except Exception as e:
            print(f"{solution}: Error occurred while loading the solution: {e}. Deleting the file and skipping.")
            os.remove(TEST_FOLDER+solution)
            continue
        try:
            if _temp.sol.mre < 1e-5:  # Check if the model converged
                print(f"{solution}: Model converged.")
                if dataset not in final_dict:
                    final_dict[dataset]={}
                if model_type not in final_dict[dataset]:
                    final_dict[dataset][model_type]=_temp
                else:
                    print(f"{solution}: Converging model already exists in final_dict. Skipping.")
            else:
                print(f"{solution}: Model did not converge.")
        except AttributeError:
            print(f"{solution}: No 'sol' attribute in the loaded data.")

batch2608_crisi_dico1_decm.pkl: Model did not converge.
batch2608_crisi_dico3_decm.pkl: Model converged.
batch2608_crisi_dico4_decm.pkl: Model converged.
batch2608_ita_elections_dico0_decm.pkl: Model did not converge.
batch2608_ita_elections_dico1_decm.pkl: Model did not converge.
batch2608_ita_elections_dico4_decm.pkl: Model converged.
batch2608_ita_elections_dico5_decm.pkl: Model converged.
batch2608_ita_elections_dico6_decm.pkl: Model converged.
batch2608_quirinale_dico0_decm.pkl: Model converged.
batch2608_quirinale_dico1_decm.pkl: Model converged.
batch2608_quirinale_dico2_decm.pkl: Model converged.
batch2608_quirinale_dico3_decm.pkl: Model converged.
batch2608_quirinale_dico4_decm.pkl: Model did not converge.
crisi_dico0_dwcm_final.pkl: Model converged.
crisi_dico0_qdecm.pkl: Model converged.
crisi_dico1_decm_conv.pkl: Model converged.
crisi_dico1_dwcm_final.pkl: Model converged.
crisi_dico1_qdecm.pkl: Model converged.
crisi_dico2_decm.pkl: Model did not converge.
crisi_dico2_dec

In [62]:
_keys=list(final_dict.keys())
_keys.sort()

In [63]:
_three_of_us=[key for key in _keys if len(final_dict[key]) == 3]
_three_of_us

['c1',
 'c2',
 'c3',
 'c4',
 'i0',
 'i2',
 'i3',
 'i4',
 'i5',
 'i6',
 'q0',
 'q1',
 'q2',
 'q3']

In [64]:
len(_three_of_us), len(_keys)

(14, 19)

In [65]:
_decm_sols=[key for key in _keys if 'DECM' in final_dict[key].keys()]
_decm_sols==_three_of_us

True